# 05 — From Token ID To Meaning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marcoharuni/forge-tokenizer/blob/main/notebooks/05_token_id_to_meaning.ipynb)

Follow the simple path from token IDs to vectors, neighbors, logits, probabilities, and samples.

**Runtime:** CPU is fine. No GPU required.

---

## 1. Install & Clone

In [ ]:
%pip install -q numpy matplotlib pytest tqdm regex
%cd /content
!test -d forge-tokenizer || git clone https://github.com/marcoharuni/forge-tokenizer.git
%cd /content/forge-tokenizer
!git pull --ff-only
%pip install -q -e .

import sys
src = '/content/forge-tokenizer/src'
if src not in sys.path:
    sys.path.insert(0, src)
print('Ready. Run the Imports cell next.')

## 2. Imports

In [ ]:
from pathlib import Path
assert Path('src/forge_tokenizer').exists(), 'Run section 1: Install & Clone first.'
import numpy as np
import regex

from forge_tokenizer import ForgeTokenizer
from forge_tokenizer.embeddings import build_cooccurrence, cosine_similarity, nearest_neighbors, one_hot, ppmi_matrix, svd_embeddings
from forge_tokenizer.unembedding import logits_to_probs, sample_from_logits, stable_softmax

print('forge-tokenizer ready')

## 3. Token IDs

In [ ]:
corpus = Path('data/tiny_corpus.txt').read_text(encoding='utf-8')
tokenizer = ForgeTokenizer().train(corpus, num_merges=100)
text = 'tokenization turns text into numbers'
ids = tokenizer.encode(text)

print(text)
print(ids)
print(tokenizer.explain(text))

## 4. One-Hot Means Address, Not Meaning

A one-hot vector mostly says which row to look up. The useful information is in the learned or constructed embedding row.

In [ ]:
idx = ids[0]
vec = one_hot(idx, tokenizer.vocab_size)
print('token id:', idx)
print('one-hot length:', len(vec))
print('nonzero position:', np.nonzero(vec)[0])

## 5. Build Tiny Distributional Embeddings

This notebook uses PPMI/SVD from the tiny corpus. It is a mechanism demo, not pretrained model quality.

In [ ]:
sentences = []
for line in corpus.lower().splitlines():
    words = regex.findall(r'\p{L}+', line)
    if words:
        sentences.append(words)

cooc, word_to_idx, idx_to_word = build_cooccurrence(sentences, window_size=2)
embeddings = svd_embeddings(ppmi_matrix(cooc), dim=8)
print('word vocabulary:', len(word_to_idx))
print('embedding shape:', embeddings.shape)

## 6. Neighbors Are Geometry

In [ ]:
query = 'tokenization'
for word, score in nearest_neighbors(embeddings, word_to_idx, idx_to_word, query, k=8):
    print(f'{word:14s} {score: .3f}')

print('cosine(tokenization, embeddings):', cosine_similarity(embeddings[word_to_idx['tokenization']], embeddings[word_to_idx['embeddings']]))

## 7. Unembedding Turns Vectors Into Logits

Here we simulate a small unembedding matrix. A real model learns this matrix during training.

In [ ]:
rng = np.random.default_rng(7)
vocab = ['token', 'text', 'number', 'embedding', 'logit']
d_model = 8
hidden_state = rng.normal(size=d_model)
unembedding = rng.normal(size=(d_model, len(vocab)))
logits = hidden_state @ unembedding
probs = stable_softmax(logits)

for token, logit, prob in zip(vocab, logits, probs):
    print(f'{token:10s} logit={logit: .3f} prob={prob: .3f}')

## 8. Sampling Is A Choice From Probabilities

In [ ]:
for seed in range(5):
    picked = sample_from_logits(logits, temperature=0.8, seed=seed)
    print(seed, vocab[picked])

print('probabilities:', logits_to_probs(logits, temperature=0.8))

---

**Next →** [06 Chat Formatting Token Tax](06_chat_formatting_token_tax.ipynb)